# Data Featurization: Using ChemBERTa

We are starting with `raw_data_vellayappan.csv` (Vellayappan et al., 2023), which contains 1,714 initial entries for Ni-catalyzed Dry Reforming of Methane (DRM). 

Our goal (in this jupyter notebook) is to clean the dataset and translate raw text catalyst names into numerical features the model can actually interpret.
We follow the following steps: 

1. Data Prep
* Drop irrelevant features and get rid of duplicates.
* Get a .txt file and Python list with the Catalyst names

2. Featurization via ChemBERTa
* **Why do we need to feautrize?** ML models can't read text strings like `10%Ni/SBA-15`. 
* **The Old Way:** One-Hot Encoding treats `10%Ni/SBA-15` and `12%Ni/SBA-15` as completely unrelated entities just because the text differs.
* **Our Way:** We use **ChemBERTa** (a chemical LLM) to convert catalyst strings into **384-dimensional vectors**. 

3. Compression via PCA
* A 384-number fingerprint is massive for this dataset size and risks overfitting. 
* We use **PCA** (Principal Component Analysis) to compress these features down to:
  * **2 components** for visual mapping.
  * **10 components** for training our machine learning models.

---
### ⚠️ IMPORTANT: ChemBERTa Domain Limitation

**ChemBERTa is trained on SMILES strings representing organic molecules.** The catalysts we use are **inorganic materials**, which are fundamentally different:

| Aspect | Organic SMILES | Your Inorganic Catalysts |
|--------|---|---|
| **Structure** | Discrete molecules with defined bonds | Metal nanoparticles on oxide supports |
| **Notation** | `C1=CC=CC=C1` (benzene) | `Ni/Al₂O₃` (metal on support) |
| **"/" meaning** | SMILES aromatic bond syntax | "supported on" (spatial arrangement) |
| **Training data** | Millions of organic molecules | ❌ No inorganic materials |

This means, when we feed our catalysts, such as `"Ni/Al2O3"`, to ChemBERTa:

✅ **What works:**
- The model recognizes element symbols (Ni, Al, O, Ce, La, etc.)
- It produces a 384-dimensional embedding

❌ **What doesn't work:**
- The model doesn't understand metal-support interfaces
- The "/" is interpreted through organic chemistry lens (wrong!)
- The embeddings are **extrapolating beyond training domain**
- Chemical similarity between inorganic catalysts is NOT captured correctly

Here are a few reasons as to why ChemBERTa may or may not be a good fit still:

Reasons it might be okay:
1. **Element recognition:** ChemBERTa knows Ni, Al, Ce are elements
2. **PCA regularization:** Compression 384D → 10D acts as denoising
3. **Data-driven learning:** Your Gaussian Process learns from actual experiments, not just embeddings
4. **Empirical validation:** If your GP R² > 0.85, embeddings are useful enough

Reasons to be cautious:
1. **Out-of-domain:** Using organic model on inorganic materials
2. **No chemical meaning:** Relationships between catalysts might be meaningless
3. **Poor generalization:** Can't reliably predict for unseen catalyst compositions
4. **Better alternatives:** Hand-crafted features specifically for inorganic materials


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import re

## Step 1: Data Prep

#### Step 1a: Cleaning Raw Dataset

In [3]:
# 1. Load your data
df = pd.read_csv("raw_data_vellayappan.csv")

# 2. Define the columns to get rid of
cols_to_remove = [
    'Paper no.', 
    'Ni Dispersion', 
    'Modifier Electronegativity'
]

# 3. Drop the columns
# 'errors=ignore' ensures the code doesn't crash if a column name has a typo
df_clean = df.drop(columns=cols_to_remove, errors='ignore')

# 4. Group by Catalyst name (sorts the rows alphabetically by the "Catalyst" column)
df_clean = df_clean.sort_values(by='Catalyst')

# 5. Save the new csv file
df_clean.to_csv("v1a_clean_ordered_data.csv", index=False)

print("Columns removed successfully.")
print(f"Remaining columns: {list(df_clean.columns)}")

Columns removed successfully.
Remaining columns: ['Catalyst', 'Ratio of CH4 in Feed', 'Reaction Temperature', 'Ni Loading', 'Reaction Time', 'Pore Size', 'Pore Volume', 'Surface Area', 'H2-TPR Peak Temperature', 'Ni Particle Size', 'GHSV', 'CH4 Conversion', 'CO2 Conversion', 'Syngas_Ratio']


#### Step 1b: Generate a .txt file and python list with the catalyst names

In [14]:
# 1. Load the clean dataset
df = pd.read_csv("v1a_clean_ordered_data.csv")

# 2. Extract initial unique names from the column
raw_unique_names = df['Catalyst'].unique()

# 3. Clean up the hidden characters and strip trailing spaces
catalyst_list = list(set([str(name).strip().replace('\xa0', '') for name in raw_unique_names]))

# 4. Save the actual pristine names to a text file
with open("catalyst_names.txt", "w", encoding="utf-8") as f:
    for name in catalyst_list:
        f.write(f"{name}\n")

# Quick sanity check
print(f"Original unique count (with hidden spaces): {len(raw_unique_names)}")
print(f"Pristine unique count (hidden spaces removed): {len(catalyst_list)}")
print("First 5 clean elements:", catalyst_list[:5])

Original unique count (with hidden spaces): 712
Pristine unique count (hidden spaces removed): 711
First 5 clean elements: ['NiZr', '10Ni/MgO', '4Ni-1ln/AL2O3', 'Ni/Mal', '0.2 K-Ni/ MgAl2O4']


#### Step 1c: Checking for any duplicates (rows that appear twice in dataset)

In [16]:
# 1. Load the dataset
df = pd.read_csv("v1a_clean_ordered_data.csv")

# Clean the 'Catalyst' column text to expose hidden duplicates
df['Catalyst'] = df['Catalyst'].astype(str).str.strip().str.replace('\xa0', '', regex=False)

# 2. Count total duplicate rows across all columns
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found (including hidden spaces): {duplicate_count}")

# 3. Display the duplicate rows (if any exist)
if duplicate_count > 0:
    print("\nHere are the duplicate rows:")
    # keep=False shows all occurrences of the duplicates so you can compare them
    print(df[df.duplicated(keep=False)].head(10)) 
else:
    print("\nYour dataset is completely free of exact duplicate rows!")

Total duplicate rows found (including hidden spaces): 2

Here are the duplicate rows:
               Catalyst  Ratio of CH4 in Feed  Reaction Temperature  \
782      Ni(15)/perlite                  0.15                 700.0   
783      Ni(15)/perlite                  0.15                 700.0   
876  Ni-PS@Ce0.9Zr0.1O2                  0.50                 600.0   
877  Ni-PS@Ce0.9Zr0.1O2                  0.50                 600.0   

     Ni Loading  Reaction Time  Pore Size  Pore Volume  Surface Area  \
782       14.80            0.1        2.7        0.020          7.00   
783       14.80            0.1        2.7        0.020          7.00   
876        3.47           20.0       16.8        0.098         29.67   
877        3.47           20.0       16.8        0.098         29.67   

     H2-TPR Peak Temperature  Ni Particle Size      GHSV  CH4 Conversion  \
782                    375.0              18.0   60000.0           69.00   
783                    375.0              18.

In [17]:
# 1. Load the dataset
df = pd.read_csv("v1a_clean_ordered_data.csv")

# Clean the 'Catalyst' column text to expose hidden duplicates
df['Catalyst'] = df['Catalyst'].astype(str).str.strip().str.replace('\xa0', '', regex=False)

# 2. Drop duplicate rows (keeps the first occurrence by default)
df_no_duplicates = df.drop_duplicates()

# 3. Save the deduplicated data to a new CSV file
df_no_duplicates.to_csv("v1c_clean_ordered_data.csv", index=False)

# Quick sanity check on rows removed
initial_rows = len(df)
final_rows = len(df_no_duplicates)
removed = initial_rows - final_rows

print(f"Removed {removed} exact duplicate rows.")
print(f"Dataset shape went from {initial_rows} rows to {final_rows} rows.")
print("Saved clean data to 'v1c_clean_ordered_data.csv'")

Removed 2 exact duplicate rows.
Dataset shape went from 1714 rows to 1712 rows.
Saved clean data to 'v1c_clean_ordered_data.csv'


#### Step 1d: Final Check

In [20]:
# 1. Load the dataset
file_name = "v1c_clean_ordered_data.csv"
df = pd.read_csv(file_name)

print("--- RUNNING DATA INTEGRITY CHECK ---")

# ==========================================
# CHECK 1: Duplicates & Hidden Whitespace
# ==========================================
# Check raw duplicates first
raw_duplicates = df.duplicated().sum()

# Check duplicates if we strip whitespaces and hidden characters again
cleaned_series = df['Catalyst'].astype(str).str.strip().str.replace('\xa0', '', regex=False)
hidden_duplicates = cleaned_series.duplicated().sum()

print(f"-> Standard duplicate rows found: {raw_duplicates}")
print(f"-> Hidden whitespace duplicate rows found: {hidden_duplicates}")


# ==========================================
# CHECK 2: Problematic Names for ChemBERTa
# ==========================================
# ChemBERTa likes standard SMILES tokens, but struggles with messy human text.
# We flag elements containing characters that often break tokenization or represent invalid inputs.
unique_catalysts = df['Catalyst'].dropna().unique()
problematic_catalysts = []

# Regex pattern to flag potential issues: 
# - Things like 'eMgO' (typos), lone punctuation like trailing commas, or weird symbols like ■, ▴, @
issue_pattern = re.compile(r'[■▴@,@_]|[a-z]M[a-z]|eM[A-Z]|\,$')

for name in unique_catalysts:
    name_str = str(name)
    
    # Flag 1: Empty names
    if name_str.strip() == "" or name_str.lower() == 'nan':
        problematic_catalysts.append((name_str, "Empty or NaN name"))
        continue
        
    # Flag 2: Strange characters or obvious human typos caught by regex
    if issue_pattern.search(name_str):
        problematic_catalysts.append((name_str, "Contains non-standard characters/typos (@, ■, ▴, trailing comma, or lower-case typing error)"))
        continue
        
    # Flag 3: Completely unreadable strings (too short to be a valid catalyst name)
    if len(name_str) <= 1:
        problematic_catalysts.append((name_str, "Too short / non-descriptive"))

# ==========================================
# 3. Report Findings and Save Text File
# ==========================================
if len(problematic_catalysts) > 0:
    print(f"\n[ALERT] Found {len(problematic_catalysts)} potentially problematic catalyst names for ChemBERTa.")
    
    # Save the flagged names to a text file for inspection
    with open("problem_catalysts_names.txt", "w", encoding="utf-8") as f:
        f.write("Flagged Catalyst Name | Reason for Flag\n")
        f.write("-" * 50 + "\n")
        for item, reason in problematic_catalysts:
            f.write(f"{item} ---> {reason}\n")
            
    print("-> Full list exported to 'problem_catalysts_names.txt'")
    print("\nTop 5 flagged examples:")
    for item, reason in problematic_catalysts[:5]:
        print(f"   • {item} ({reason})")
else:
    print("\n[SUCCESS] No problematic catalyst names detected. Everything looks safe for ChemBERTa!")

--- RUNNING DATA INTEGRITY CHECK ---
-> Standard duplicate rows found: 0
-> Hidden whitespace duplicate rows found: 1001

[ALERT] Found 83 potentially problematic catalyst names for ChemBERTa.
-> Full list exported to 'problem_catalysts_names.txt'

Top 5 flagged examples:
   • 10% Ni/CeO2 eMgO (Contains non-standard characters/typos (@, ■, ▴, trailing comma, or lower-case typing error))
   • 12%Ni@Al2O3 (Contains non-standard characters/typos (@, ■, ▴, trailing comma, or lower-case typing error))
   • 12NiMgAl-S (Contains non-standard characters/typos (@, ■, ▴, trailing comma, or lower-case typing error))
   • 12NiMgO-S (Contains non-standard characters/typos (@, ■, ▴, trailing comma, or lower-case typing error))
   • 12NiMgO/Al2O3 (Contains non-standard characters/typos (@, ■, ▴, trailing comma, or lower-case typing error))


#### Step 1e: Fix the problematic catalyst names

In [21]:
# 1. Load your dataset 
df = pd.read_csv("v1c_clean_ordered_data.csv")

# 2. Define the translation function for ChemBERTa compatibility
def clean_catalyst_for_chemberta(name):
    if pd.isna(name):
        return name
    
    name_str = str(name).strip()
    
    # Remove trailing commas or punctuation
    name_str = re.sub(r'[,\s_]+$', '', name_str)
    
    # Remove geometric symbols used as plot markers
    name_str = name_str.replace('■', '').replace('▴', '')
    
    # Standardize core spaces (e.g., "Ni @ Si" -> "Ni@Si")
    name_str = re.sub(r'\s*@\s*', '@', name_str)
    
    # Replace core '@' character with a hyphen '-' so the tokenizer parses it as a composite material
    name_str = name_str.replace('@', '-')
    
    # Clean up spaces around slashes (e.g., "Ni / MgAl" -> "Ni/MgAl")
    name_str = re.sub(r'\s*/\s*', '/', name_str)
    
    # Clean up dual hyphens or trailing weird spaces
    name_str = re.sub(r'-+', '-', name_str)
    name_str = re.sub(r'\s+', ' ', name_str)
    
    return name_str.strip()

# 3. Apply the transformation to the Catalyst column
df['Catalyst_Cleaned'] = df['Catalyst'].apply(clean_catalyst_for_chemberta)

# 4. Overwrite original column or keep both for tracking
df['Catalyst'] = df['Catalyst_Cleaned']
df = df.drop(columns=['Catalyst_Cleaned'])

# 5. Drop any newly exposed duplicates now that names are standardized
df = df.drop_duplicates()

# 6. Save the perfectly compliant dataset
df.to_csv("v1e_chemberta_ready_data.csv", index=False)

# 7. Extract the pristine list for the active notebook session
catalyst_list = list(df['Catalyst'].unique())

print("--- POST-CLEANING SUMMARY ---")
print(f"Total unique catalysts ready for ChemBERTa: {len(catalyst_list)}")
print("\nVerifying a few transformed examples:")
examples = ["12%Ni@Al2O3", "Ce (10)-Ni(15)/perlite,", "NCMZ (■)", "Ni @ Si @ CNF"]
for ex in examples:
    print(f"   Original: {ex}  ===>  Cleaned: {clean_catalyst_for_chemberta(ex)}")

--- POST-CLEANING SUMMARY ---
Total unique catalysts ready for ChemBERTa: 709

Verifying a few transformed examples:
   Original: 12%Ni@Al2O3  ===>  Cleaned: 12%Ni-Al2O3
   Original: Ce (10)-Ni(15)/perlite,  ===>  Cleaned: Ce (10)-Ni(15)/perlite
   Original: NCMZ (■)  ===>  Cleaned: NCMZ ()
   Original: Ni @ Si @ CNF  ===>  Cleaned: Ni-Si-CNF


## Step 2: Data Featurization

#### Step 2a: Generating Chemical Embeddings with ChemBERTa
The code below translates our text-based catalyst strings into numerical coordinates (vectors) using a transformer model, compresses that multi-dimensional data, and plots a "Chemical Space Map."

##### How the Code Works:

1. **Loading the Chemistry LLM:** We initialize **ChemBERTa-77M-MLM** (via the Hugging Face Transformers library). This model has read millions of chemical formulas, meaning it implicitly understands relations between metals, supports, and ratios.
2. **Batch Processing Function (`get_embeddings_batched`):** * To prevent our computer's memory from crashing, we split our long list of catalysts into small batches of 32.
   * The text is tokenized, passed through ChemBERTa, and we take the mathematical average (`mean(dim=1)`) of the output layers. This yields a single **384-dimensional fingerprint vector** for every catalyst in the list.
3. **Dimensionality Reduction (PCA):** A 384-dimensional space contains too much noise for our dataset size. We apply Principal Component Analysis (PCA) to capture the highest variance and compress those 384 numbers down to 10 principal components, creating a dense, robust feature matrix.

In [22]:
# 1. Load the ChemBERTa model
model_name = "DeepChem/ChemBERTa-77M-MLM"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Define the embedding function
def get_embeddings_batched(text_list, batch_size=32):
    all_embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            # Take the mean of the token embeddings
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings.numpy())
    return np.vstack(all_embeddings)

# 3. Generate the raw 384-dimensional chemical embeddings
print("Extracting chemical fingerprints using ChemBERTa...")
X_raw = get_embeddings_batched(catalyst_list)

# 4. Compress the feature space to 10 components for your ML model
print("Compressing feature space to 10 principal components...")
pca_ml = PCA(n_components=10)
X_ml_features = pca_ml.fit_transform(X_raw)

# Quick verification
print(f"\nSuccess! Generated ML feature matrix shape: {X_ml_features.shape}")
print("Your 'X_ml_features' array is ready to be used as input for your ML models.")

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 6641.71it/s]
[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extracting chemical fingerprints using ChemBERTa...
Compressing feature space to 10 principal components...

Success! Generated ML feature matrix shape: (709, 10)
Your 'X_ml_features' array is ready to be used as input for your ML models.


#### Step 2b: Dimensionality Reduction and Feature Matrix Export

In [27]:
# 1. Run PCA with 10 components for better accuracy
pca_final = PCA(n_components=10)
X_final = pca_final.fit_transform(X_raw)

# 2. Create a clean DataFrame with components
feature_names = [f"PC{i+1}" for i in range(10)]
df_features = pd.DataFrame(X_final, columns=feature_names)

# 3. Explicitly insert the Catalyst name as the first column
df_features.insert(0, 'Catalyst', catalyst_list)

# 4. Save it to a CSV file (index=False because Catalyst is now a proper column)
df_features.to_csv("v2b_catalyst_featurized.csv", index=False)

print("--- TOP 5 CATALYSTS FEATURIZED ---")
print(df_features.head())
print("\nFile 'v2b_catalyst_featurized.csv' created.")

--- TOP 5 CATALYSTS FEATURIZED ---
                          Catalyst       PC1       PC2       PC3       PC4  \
0     0.05 g/g (5 mass %) Ni/Al2O3 -0.879889  0.405558 -0.017228 -0.835636   
1  0.075 g/g (7.5 mass %) Ni/Al2O3 -1.140585  0.546000  0.681388 -1.278484   
2     0.1 g/g (10 mass %) Ni/Al2O3 -1.167795  0.312266  0.475875 -1.070202   
3                 0.2 K-Ni/MgAl2O4  0.398629  1.395119 -0.201752  0.148200   
4                    0.2Cu-Ni/SiO2 -0.893367  0.750368 -1.492172 -0.633350   

        PC5       PC6       PC7       PC8       PC9      PC10  
0  0.321209 -0.032130 -0.396534  0.452465  0.494825 -0.285596  
1  0.974996  0.327887 -0.437630  0.403803  0.285239 -0.577713  
2  0.094437 -0.431905 -1.064680  0.477327  0.595488 -0.627575  
3  0.827779 -0.214399  0.452386 -0.204365  0.629000  0.413755  
4  0.190954  0.717162 -0.871331  0.058366 -0.324016  0.229283  

File 'v2b_catalyst_featurized.csv' created.


#### Step 2c: Feature Merging & Matrix Finalization

In [ ]:
# 1. Load the data files
file_exp = "v1e_chemberta_ready_data.csv"
file_feat = "v2b_catalyst_featurized.csv"

df_exp = pd.read_csv(file_exp)
df_feat = pd.read_csv(file_feat)

print("--- RUNNING ROBUST INTEGRITY & CONVERSION CHECK ---")

# Extract unique identifiers for verification
exp_unique = set(df_exp['Catalyst'].dropna().unique())
feat_unique = set(df_feat['Catalyst'].dropna().unique())

missing_features = exp_unique - feat_unique
extra_features = feat_unique - exp_unique

print(f"-> Total unique catalysts in experimental dataset: {len(exp_unique)}")
print(f"-> Total unique catalysts found in featurized file: {len(feat_unique)}")
print(f"-> Missing PCA entries for experimental data: {len(missing_features)}")

# Robust numeric validation check across the PCA dimensions
pca_cols = [f"PC{i}" for i in range(1, 11)]
nan_counts = df_feat[pca_cols].isna().sum().sum()
inf_counts = np.isinf(df_feat[pca_cols]).sum().sum()

print(f"-> NaN values detected in PCA dimensions: {nan_counts}")
print(f"-> Infinite/Overflow flags detected in PCA dimensions: {inf_counts}")

# =========================================================
# 2. Handle Errors / Missing Features Log Exception
# =========================================================
if len(missing_features) > 0:
    print(f"\n[ALERT] Found {len(missing_features)} unmapped catalysts! Writing error log...")
    with open("missing_pca_errors.txt", "w", encoding="utf-8") as f:
        f.write("ERROR LOG: The following catalysts contain experimental records but have missing PCA coordinates:\n")
        for catalyst in sorted(list(missing_features)):
            f.write(f"- {catalyst}\n")
    print("-> Error log saved successfully to 'missing_pca_errors.txt'.")
else:
    print("\n[SUCCESS] Conversion verification passed! Perfect matching alignment achieved.")
    with open("missing_pca_errors.txt", "w", encoding="utf-8") as f:
        f.write("No errors found. Every catalyst has successfully mapped to its corresponding 10 PCA vectors.\n")

# =========================================================
# 3. Precise Merging & Text-to-Vector Feature Replacement
# =========================================================
# left-merge matches every experimental row to its vector fingerprint
df_merged = pd.merge(df_exp, df_feat, on="Catalyst", how="left")

# Reorder columns to put original columns first (minus text 'Catalyst') and PCA columns at the back
experimental_numerical_cols = [col for col in df_exp.columns if col != 'Catalyst']
final_column_layout = experimental_numerical_cols + pca_cols
df_final = df_merged[final_column_layout]

# 4. Export the clean ML feature matrix
output_file = "v2c_featurized_ChemBERTa.csv"
df_final.to_csv(output_file, index=False)

print(f"\n--- EXPORT COMPLETED ---")
print(f"Master CSV file generated: '{output_file}'")
print(f"Final Data Matrix Shape: {df_final.shape} rows/columns")
print("\nFinal Column List Verification:")
print(list(df_final.columns))
print("\nPreview of the first 3 rows:")
print(df_final.head(3))

--- RUNNING ROBUST INTEGRITY & CONVERSION CHECK ---
-> Total unique catalysts in experimental dataset: 709
-> Total unique catalysts found in featurized file: 709
-> Missing PCA entries for experimental data: 0
-> NaN values detected in PCA dimensions: 0
-> Infinite/Overflow flags detected in PCA dimensions: 0

[SUCCESS] Conversion verification passed! Perfect matching alignment achieved.

--- EXPORT COMPLETED ---
Master CSV file generated: 'v2c_featurized_ml_ready.csv'
Final Data Matrix Shape: (1712, 23) rows/columns

Final Column List Verification:
['Ratio of CH4 in Feed', 'Reaction Temperature', 'Ni Loading', 'Reaction Time', 'Pore Size', 'Pore Volume', 'Surface Area', 'H2-TPR Peak Temperature', 'Ni Particle Size', 'GHSV', 'CH4 Conversion', 'CO2 Conversion', 'Syngas_Ratio', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']

Preview of the first 3 rows:
   Ratio of CH4 in Feed  Reaction Temperature  Ni Loading  Reaction Time  \
0                   0.5            